In [1]:
import os, random, time, json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from imblearn.metrics import specificity_score
from mambapy.vim import VMamba, MambaConfig
from thop import profile

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [2]:
AUG_ROOT = "D:/mamba_model/aug_clean_tio"        
TAG      = "tio"

COHORT_CSV = "D:/mamba_model/thesis_cohort_clean.csv"
MRI_CACHE  = f"{AUG_ROOT}/wb_mri"
PET_CACHE  = f"{AUG_ROOT}/wb_pet"
CKPT_DIR   = f"D:/mamba_model/checkpoints_v7_wb_{TAG}"
RESULTS    = f"D:/mamba_model/v7_wb_{TAG}_results.json"
os.makedirs(CKPT_DIR, exist_ok=True)

SPLIT_SEED  = 42
AUG_SEEDS   = [1, 101, 42]
BATCH_SIZE  = 1
BRAIN_SIZE  = 256
NUM_WORKERS = 0

print(f"aug:  {AUG_ROOT}")
print(f"ckpt: {CKPT_DIR}")
for p in (MRI_CACHE, PET_CACHE):
    n = len(os.listdir(p)) if os.path.isdir(p) else 0
    print(f"  {os.path.basename(p)}: {n} files{'  *** MISSING ***' if n == 0 else ''}")
print(f"\ntokens per volume: {(BRAIN_SIZE // 8) ** 3:,}")

aug:  D:/mamba_model/aug_clean_tio
ckpt: D:/mamba_model/checkpoints_v7_wb_tio
  wb_mri: 560 files
  wb_pet: 560 files

tokens per volume: 32,768


In [3]:
class VimEncoder(nn.Module):
    """Bidirectional Mamba over a token sequence. No CNN, no pretraining."""
    def __init__(self, d_model=32, n_layers=2, d_state=16):
        super().__init__()
        cfg = MambaConfig(d_model=d_model, n_layers=n_layers, d_state=d_state,
                          bidirectional=True, divide_output=True,
                          pscan=True, use_cuda=False)
        self.encoder = VMamba(cfg)
        self.final_norm = nn.LayerNorm(d_model)
    def forward(self, tokens):
        return self.final_norm(self.encoder(tokens))

In [4]:
class WholeBrainPatchEmbed3D(nn.Module):
    """256^3 volume -> non-overlapping 8^3 patches -> one token each (32,768)."""
    def __init__(self, brain_size=BRAIN_SIZE, patch_size=8, d_model=32):
        super().__init__()
        self.patch_size = patch_size
        self.grid_size  = brain_size // patch_size
        self.n_tokens   = self.grid_size ** 3
        self.d_model    = d_model

        self.patch_conv   = nn.Conv3d(1, d_model, kernel_size=patch_size, stride=patch_size)
        self.depth_embed  = nn.Embedding(self.grid_size, d_model)
        self.height_embed = nn.Embedding(self.grid_size, d_model)
        self.width_embed  = nn.Embedding(self.grid_size, d_model)
        with torch.no_grad():
            for e in [self.depth_embed, self.height_embed, self.width_embed]:
                e.weight.mul_(0.02)

        d, h, w = torch.meshgrid(torch.arange(self.grid_size), torch.arange(self.grid_size),
                                 torch.arange(self.grid_size), indexing="ij")
        self.register_buffer("coordinates", torch.stack([d, h, w], -1).reshape(-1, 3),
                             persistent=False)

    def forward(self, volume):
        tokens = self.patch_conv(volume).flatten(2).transpose(1, 2)
        c = self.coordinates
        spatial = (self.depth_embed(c[:, 0]) + self.height_embed(c[:, 1])
                   + self.width_embed(c[:, 2]))
        tokens = tokens + spatial[None, :, :]
        occ = F.max_pool3d((volume.abs() > 1e-6).float(),
                           kernel_size=self.patch_size, stride=self.patch_size)
        valid = occ.flatten(1).bool()
        return tokens * valid.unsqueeze(-1).to(tokens.dtype), valid


class WholeBrainBranch(nn.Module):
    def __init__(self, brain_size=BRAIN_SIZE, patch_size=8, d_model=32,
                 n_layers=2, d_state=16):
        super().__init__()
        self.patch_embed = WholeBrainPatchEmbed3D(brain_size, patch_size, d_model)
        self.vim = VimEncoder(d_model, n_layers, d_state)
    def forward(self, volume):
        tokens, valid = self.patch_embed(volume)
        tokens = self.vim(tokens)
        w = valid.unsqueeze(-1).to(tokens.dtype)
        return (tokens * w).sum(dim=1) / w.sum(dim=1).clamp_min(1.0)


class VisionMambaModel(nn.Module):
    def __init__(self, brain_size=BRAIN_SIZE, patch_size=8, d_model=32,
                 n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.branch     = WholeBrainBranch(brain_size, patch_size, d_model, n_layers, d_state)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, n_classes)
    def forward(self, volume):
        return self.classifier(self.dropout(self.branch(volume)))


class MultimodalVisionMambaModel(nn.Module):
    def __init__(self, brain_size=BRAIN_SIZE, patch_size=8, d_model=32,
                 n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.mri_branch = WholeBrainBranch(brain_size, patch_size, d_model, n_layers, d_state)
        self.pet_branch = WholeBrainBranch(brain_size, patch_size, d_model, n_layers, d_state)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model * 2, n_classes)
    def forward(self, mri_vol, pet_vol):
        f = torch.cat([self.mri_branch(mri_vol), self.pet_branch(pet_vol)], dim=1)
        return self.classifier(self.dropout(f))

In [5]:
df = pd.read_csv(COHORT_CSV)
sessions, labels = df["mri_session"].values, df["outcome_label"].values

X_tv, X_test, y_tv, y_test = train_test_split(
    sessions, labels, test_size=0.2, random_state=SPLIT_SEED, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, random_state=SPLIT_SEED, stratify=y_tv)

session_to_subject = dict(zip(df["mri_session"], df["subject_id"]))
print(f"train {len(X_train)} | val {len(X_val)} | test {len(X_test)} "
      f"| test pos {int(y_test.sum())}")


class WholeBrainDataset(Dataset):
    """Streams from disk -- no RAM cache, volumes are 67 MB each."""
    def __init__(self, sessions, labels, cache_dir, is_mri=True, is_train=False):
        self.samples, self.cache_dir = [], cache_dir
        for ses, lab in zip(sessions, labels):
            key = ses if is_mri else session_to_subject[ses]
            self.samples.append((key, lab, "orig"))
            if is_train:
                for s in AUG_SEEDS:
                    self.samples.append((key, lab, f"aug{s}"))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        key, lab, ver = self.samples[i]
        a = np.load(f"{self.cache_dir}/{key}_{ver}.npy").astype(np.float32)
        return torch.from_numpy(a).unsqueeze(0), torch.tensor(lab, dtype=torch.long), key


class MultimodalWholeBrainDataset(Dataset):
    def __init__(self, sessions, labels, mri_dir, pet_dir, is_train=False):
        self.samples, self.mri_dir, self.pet_dir = [], mri_dir, pet_dir
        for ses, lab in zip(sessions, labels):
            sid = session_to_subject[ses]
            self.samples.append((ses, sid, lab, "orig"))
            if is_train:
                for s in AUG_SEEDS:
                    self.samples.append((ses, sid, lab, f"aug{s}"))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        mk, pk, lab, ver = self.samples[i]
        m = np.load(f"{self.mri_dir}/{mk}_{ver}.npy").astype(np.float32)
        p = np.load(f"{self.pet_dir}/{pk}_{ver}.npy").astype(np.float32)
        return (torch.from_numpy(m).unsqueeze(0), torch.from_numpy(p).unsqueeze(0),
                torch.tensor(lab, dtype=torch.long), mk)


def dl(ds, shuffle=False):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      persistent_workers=(NUM_WORKERS > 0))

mri_loaders = (dl(WholeBrainDataset(X_train, y_train, MRI_CACHE, True, True), True),
               dl(WholeBrainDataset(X_val,   y_val,   MRI_CACHE, True, False)),
               dl(WholeBrainDataset(X_test,  y_test,  MRI_CACHE, True, False)))

pet_loaders = (dl(WholeBrainDataset(X_train, y_train, PET_CACHE, False, True), True),
               dl(WholeBrainDataset(X_val,   y_val,   PET_CACHE, False, False)),
               dl(WholeBrainDataset(X_test,  y_test,  PET_CACHE, False, False)))

mm_loaders  = (dl(MultimodalWholeBrainDataset(X_train, y_train, MRI_CACHE, PET_CACHE, True), True),
               dl(MultimodalWholeBrainDataset(X_val,   y_val,   MRI_CACHE, PET_CACHE, False)),
               dl(MultimodalWholeBrainDataset(X_test,  y_test,  MRI_CACHE, PET_CACHE, False)))

print(f"train samples (4x augmented): {len(mri_loaders[0].dataset)}")

t0 = time.time()
for i, _ in enumerate(mri_loaders[0]):
    if i >= 20: break
el = time.time() - t0
print(f"20 volumes loaded in {el:.1f}s "
      f"-> ~{el/20*len(mri_loaders[0].dataset)/60:.0f} min/epoch on I/O alone")

train 120 | val 40 | test 40 | test pos 20
train samples (4x augmented): 480
20 volumes loaded in 9.3s -> ~4 min/epoch on I/O alone


In [6]:
def train_epoch(model, loader, opt, crit, mm):
    model.train(); tot = 0
    for batch in loader:
        opt.zero_grad()
        if mm:
            a, b, lb, _ = batch; out = model(a.to(device), b.to(device))
        else:
            a, lb, _ = batch;    out = model(a.to(device))
        loss = crit(out, lb.to(device)); loss.backward(); opt.step(); tot += loss.item()
    return tot / len(loader)

def evaluate(model, loader, crit, mm):
    model.eval(); tot, P, L = 0, [], []
    with torch.no_grad():
        for batch in loader:
            if mm:
                a, b, lb, _ = batch; out = model(a.to(device), b.to(device))
            else:
                a, lb, _ = batch;    out = model(a.to(device))
            tot += crit(out, lb.to(device)).item()
            P.extend(out.argmax(1).cpu().numpy()); L.extend(lb.numpy())
    return (tot / len(loader), np.mean(np.array(P) == np.array(L)),
            recall_score(L, P, zero_division=0), specificity_score(L, P))

def measure_inference(model, loader, mm, n=20):
    model.eval(); ts = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n: break
            if mm:
                a, b = batch[0].to(device), batch[1].to(device); bs = a.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()
                t0 = time.time(); _ = model(a, b)
            else:
                a = batch[0].to(device); bs = a.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()
                t0 = time.time(); _ = model(a)
            if device.type == 'cuda': torch.cuda.synchronize()
            ts.append((time.time() - t0) / bs)
    return np.mean(ts), np.std(ts)

def compute_flops(model, loader, mm):
    try:
        model.eval(); b = next(iter(loader))
        with torch.no_grad():
            inp = (b[0][:1].to(device), b[1][:1].to(device)) if mm else (b[0][:1].to(device),)
            macs, _ = profile(model, inputs=inp, verbose=False)
        return macs * 2
    except Exception as e:
        print(f"  (FLOPs failed: {e})"); return None


def run_seed(seed, model_cls, loaders, mm, prefix,
             max_epochs=101, patience=15, min_epochs=25, lr=1e-4, log_every=1):
    torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    np.random.seed(seed); random.seed(seed)
    tr, va, te = loaders

    model = model_cls(brain_size=BRAIN_SIZE, d_model=32, n_layers=2,
                      n_classes=2, dropout=0.4).to(device)
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    opt  = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    sch  = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=10)

    best, no_imp, best_ep, total = float('inf'), 0, 0, 0
    path = f"{CKPT_DIR}/{prefix}_seed{seed}.pt"
    print(f"\n--- {prefix} seed {seed} ---")

    for ep in range(1, max_epochs):
        t0 = time.time()
        trl = train_epoch(model, tr, opt, crit, mm)
        vl, vacc, vtpr, vtnr = evaluate(model, va, crit, mm)
        sch.step(vl); dt = time.time() - t0; total += dt
        if ep % log_every == 0 or ep == 1:
            print(f"  ep {ep:>3} | train {trl:.4f} | val {vl:.4f} | "
                  f"acc {vacc:.3f} tpr {vtpr:.3f} tnr {vtnr:.3f} | {dt:.0f}s "
                  f"| cum {total/60:.0f}m")
        if vl < best:
            best, best_ep, no_imp = vl, ep, 0
            torch.save(model.state_dict(), path)
        else:
            no_imp += 1
            if ep >= min_epochs and no_imp >= patience:
                print(f"  early stop {ep}, best {best_ep}"); break

    if best_ep < 5:
        print(f"  WARNING: best epoch {best_ep} -- may never have left initialisation")

    model.load_state_dict(torch.load(path, weights_only=True))
    _, acc, tpr, tnr = evaluate(model, te, crit, mm)
    npar = sum(p.numel() for p in model.parameters() if p.requires_grad)
    inf_m, inf_s = measure_inference(model, te, mm)
    fl = compute_flops(model, te, mm)

    print(f"  >>> TEST Acc={acc*100:.1f}% TPR={tpr*100:.1f}% TNR={tnr*100:.1f}% | "
          f"params={npar:,} train={total/60:.1f}min inf={inf_m*1000:.2f}ms "
          f"{f'{fl/1e9:.2f}GFLOPs' if fl else ''} best_ep={best_ep}")

    return {"seed": seed, "acc": acc, "tpr": tpr, "tnr": tnr, "best_epoch": best_ep,
            "train_time_sec": total, "n_params": npar,
            "inf_time_ms": inf_m * 1000, "inf_std_ms": inf_s * 1000, "flops": fl}


results = {"mri": [], "pet": [], "mm": []}
print("ready")

ready


In [7]:
results["mri"].append(run_seed(1, VisionMambaModel, mri_loaders, False, "v7_wb_mri"))


--- v7_wb_mri seed 1 ---
  ep   1 | train 0.6976 | val 0.6955 | acc 0.500 tpr 1.000 tnr 0.000 | 309s | cum 5m
  ep   2 | train 0.6940 | val 0.6925 | acc 0.575 tpr 0.550 tnr 0.600 | 308s | cum 10m
  ep   3 | train 0.6946 | val 0.6919 | acc 0.525 tpr 1.000 tnr 0.050 | 309s | cum 15m
  ep   4 | train 0.6903 | val 0.6908 | acc 0.475 tpr 0.000 tnr 0.950 | 308s | cum 21m
  ep   5 | train 0.6907 | val 0.6890 | acc 0.450 tpr 0.000 tnr 0.900 | 308s | cum 26m
  ep   6 | train 0.6893 | val 0.6873 | acc 0.550 tpr 0.900 tnr 0.200 | 309s | cum 31m
  ep   7 | train 0.6836 | val 0.6878 | acc 0.550 tpr 0.900 tnr 0.200 | 306s | cum 36m
  ep   8 | train 0.6869 | val 0.6847 | acc 0.550 tpr 0.800 tnr 0.300 | 309s | cum 41m
  ep   9 | train 0.6860 | val 0.6829 | acc 0.700 tpr 0.700 tnr 0.700 | 307s | cum 46m
  ep  10 | train 0.6806 | val 0.6813 | acc 0.475 tpr 0.000 tnr 0.950 | 307s | cum 51m
  ep  11 | train 0.6789 | val 0.6793 | acc 0.575 tpr 0.750 tnr 0.400 | 307s | cum 56m
  ep  12 | train 0.6770 | val

In [8]:
results["pet"].append(run_seed(1, VisionMambaModel, pet_loaders, False, "v7_wb_pet"))


--- v7_wb_pet seed 1 ---
  ep   1 | train 0.6962 | val 0.6861 | acc 0.500 tpr 1.000 tnr 0.000 | 332s | cum 6m
  ep   2 | train 0.6929 | val 0.6808 | acc 0.600 tpr 0.650 tnr 0.550 | 316s | cum 11m
  ep   3 | train 0.6907 | val 0.6775 | acc 0.575 tpr 0.950 tnr 0.200 | 315s | cum 16m
  ep   4 | train 0.6864 | val 0.6806 | acc 0.625 tpr 0.300 tnr 0.950 | 315s | cum 21m
  ep   5 | train 0.6887 | val 0.6748 | acc 0.700 tpr 0.450 tnr 0.950 | 321s | cum 27m
  ep   6 | train 0.6839 | val 0.6676 | acc 0.525 tpr 1.000 tnr 0.050 | 324s | cum 32m
  ep   7 | train 0.6734 | val 0.6680 | acc 0.500 tpr 0.850 tnr 0.150 | 316s | cum 37m
  ep   8 | train 0.6761 | val 0.6665 | acc 0.650 tpr 0.800 tnr 0.500 | 318s | cum 43m
  ep   9 | train 0.6693 | val 0.6562 | acc 0.775 tpr 0.700 tnr 0.850 | 317s | cum 48m
  ep  10 | train 0.6544 | val 0.6235 | acc 0.725 tpr 0.650 tnr 0.800 | 317s | cum 53m
  ep  11 | train 0.6395 | val 0.6300 | acc 0.725 tpr 0.600 tnr 0.850 | 318s | cum 58m
  ep  12 | train 0.6287 | val

In [9]:
results["mm"].append(run_seed(1, MultimodalVisionMambaModel, mm_loaders, True, "v7_wb_mm"))


--- v7_wb_mm seed 1 ---
  ep   1 | train 0.6944 | val 0.6842 | acc 0.575 tpr 0.950 tnr 0.200 | 632s | cum 11m
  ep   2 | train 0.6909 | val 0.6842 | acc 0.575 tpr 0.950 tnr 0.200 | 618s | cum 21m
  ep   3 | train 0.6827 | val 0.6866 | acc 0.500 tpr 0.000 tnr 1.000 | 617s | cum 31m
  ep   4 | train 0.6843 | val 0.6759 | acc 0.575 tpr 0.950 tnr 0.200 | 619s | cum 41m
  ep   5 | train 0.6853 | val 0.6811 | acc 0.525 tpr 0.100 tnr 0.950 | 620s | cum 52m
  ep   6 | train 0.6782 | val 0.6858 | acc 0.500 tpr 0.000 tnr 1.000 | 620s | cum 62m
  ep   7 | train 0.6774 | val 0.6624 | acc 0.525 tpr 0.800 tnr 0.250 | 621s | cum 72m
  ep   8 | train 0.6666 | val 0.6510 | acc 0.750 tpr 0.600 tnr 0.900 | 620s | cum 83m
  ep   9 | train 0.6464 | val 0.6299 | acc 0.675 tpr 0.800 tnr 0.550 | 618s | cum 93m
  ep  10 | train 0.6436 | val 0.6199 | acc 0.725 tpr 0.650 tnr 0.800 | 619s | cum 103m
  ep  11 | train 0.6239 | val 0.5917 | acc 0.725 tpr 0.650 tnr 0.800 | 620s | cum 114m
  ep  12 | train 0.6350 | v

In [10]:
INCLUDE_SEEDS = [1]

def summarize(rs, name, include=INCLUDE_SEEDS):
    rs = [r for r in rs if r['seed'] in include]
    if not rs:
        print(f"{name}: no runs"); return
    a  = [r['acc'] for r in rs]; t = [r['tpr'] for r in rs]; n = [r['tnr'] for r in rs]
    tm = [r['train_time_sec'] for r in rs]; inf = [r['inf_time_ms'] for r in rs]
    fl = [r['flops'] for r in rs if r['flops']]
    sd = lambda v: np.std(v, ddof=1) * 100 if len(v) > 1 else 0.0
    print(f"{name}: Acc={np.mean(a)*100:.1f}±{sd(a):.1f}% | "
          f"TPR={np.mean(t)*100:.1f}±{sd(t):.1f}% | "
          f"TNR={np.mean(n)*100:.1f}±{sd(n):.1f}% | "
          f"Params={rs[0]['n_params']:,} | Train={np.mean(tm)/60:.1f}m | "
          f"Inf={np.mean(inf):.2f}ms | "
          f"{f'{np.mean(fl)/1e9:.2f}GFLOPs' if fl else 'N/A'} "
          f"| seeds={[r['seed'] for r in rs]}")

print(f"=== v7 whole-brain Vision Mamba — {TAG} augmentation, 200 subjects ===")
print(f"    native 256^3, {(BRAIN_SIZE//8)**3:,} tokens, brain-masked (mask.mgz)")
print(f"    seeds {INCLUDE_SEEDS}\n")
for k, n in [('mri', 'MRI-only  '), ('pet', 'PET-only  '), ('mm', 'Multimodal')]:
    summarize(results[k], n)

print("\nNOTE: whole-brain streams from disk; ROI models train from an in-memory")

with open(RESULTS, 'w') as f:
    json.dump({k: [{kk: (float(vv) if isinstance(vv, (float, np.floating)) else vv)
                    for kk, vv in r.items()} for r in v] for k, v in results.items()},
              f, indent=2)
print(f"\nsaved {RESULTS} (all seeds retained)")

=== v7 whole-brain Vision Mamba — tio augmentation, 200 subjects ===
    native 256^3, 32,768 tokens, brain-masked (mask.mgz)
    seeds [1]

MRI-only  : Acc=70.0±0.0% | TPR=75.0±0.0% | TNR=65.0±0.0% | Params=47,074 | Train=297.9m | Inf=72.90ms | 2.52GFLOPs | seeds=[1]
PET-only  : Acc=62.5±0.0% | TPR=50.0±0.0% | TNR=75.0±0.0% | Params=47,074 | Train=395.1m | Inf=70.80ms | 2.52GFLOPs | seeds=[1]
Multimodal: Acc=62.5±0.0% | TPR=55.0±0.0% | TNR=70.0±0.0% | Params=94,146 | Train=567.9m | Inf=102.63ms | 5.05GFLOPs | seeds=[1]

NOTE: whole-brain streams from disk; ROI models train from an in-memory

saved D:/mamba_model/v7_wb_tio_results.json (all seeds retained)


In [11]:
results["mri"].append(run_seed(7, VisionMambaModel, mri_loaders, False, "v7_wb_mri"))


--- v7_wb_mri seed 7 ---
  ep   1 | train 0.6957 | val 0.6928 | acc 0.525 tpr 0.950 tnr 0.100 | 307s | cum 5m
  ep   2 | train 0.6952 | val 0.6918 | acc 0.500 tpr 0.600 tnr 0.400 | 308s | cum 10m
  ep   3 | train 0.6903 | val 0.6906 | acc 0.550 tpr 0.900 tnr 0.200 | 307s | cum 15m
  ep   4 | train 0.6896 | val 0.6893 | acc 0.550 tpr 0.900 tnr 0.200 | 307s | cum 20m
  ep   5 | train 0.6917 | val 0.6879 | acc 0.550 tpr 0.900 tnr 0.200 | 307s | cum 26m
  ep   6 | train 0.6866 | val 0.6876 | acc 0.550 tpr 0.900 tnr 0.200 | 307s | cum 31m
  ep   7 | train 0.6855 | val 0.6834 | acc 0.550 tpr 0.200 tnr 0.900 | 308s | cum 36m
  ep   8 | train 0.6847 | val 0.6845 | acc 0.500 tpr 0.000 tnr 1.000 | 308s | cum 41m
  ep   9 | train 0.6840 | val 0.6862 | acc 0.550 tpr 1.000 tnr 0.100 | 307s | cum 46m
  ep  10 | train 0.6795 | val 0.6811 | acc 0.500 tpr 0.000 tnr 1.000 | 307s | cum 51m
  ep  11 | train 0.6790 | val 0.6766 | acc 0.525 tpr 0.750 tnr 0.300 | 308s | cum 56m
  ep  12 | train 0.6798 | val

KeyboardInterrupt: 

In [ ]:
results["mri"].append(run_seed(123, VisionMambaModel, mri_loaders, False, "v7_wb_mri"))

In [ ]:
results["pet"].append(run_seed(7, VisionMambaModel, pet_loaders, False, "v7_wb_pet"))

In [ ]:
results["pet"].append(run_seed(123, VisionMambaModel, pet_loaders, False, "v7_wb_pet"))

In [ ]:
results["mm"].append(run_seed(7, MultimodalVisionMambaModel, mm_loaders, True, "v7_wb_mm"))

In [ ]:
results["mm"].append(run_seed(123, MultimodalVisionMambaModel, mm_loaders, True, "v7_wb_mm"))

In [ ]:
INCLUDE_SEEDS = [1, 7, 123]

def summarize(rs, name, include=INCLUDE_SEEDS):
    rs = [r for r in rs if r['seed'] in include]
    if not rs:
        print(f"{name}: no runs"); return
    a  = [r['acc'] for r in rs]; t = [r['tpr'] for r in rs]; n = [r['tnr'] for r in rs]
    tm = [r['train_time_sec'] for r in rs]; inf = [r['inf_time_ms'] for r in rs]
    fl = [r['flops'] for r in rs if r['flops']]
    sd = lambda v: np.std(v, ddof=1) * 100 if len(v) > 1 else 0.0
    print(f"{name}: Acc={np.mean(a)*100:.1f}±{sd(a):.1f}% | "
          f"TPR={np.mean(t)*100:.1f}±{sd(t):.1f}% | "
          f"TNR={np.mean(n)*100:.1f}±{sd(n):.1f}% | "
          f"Params={rs[0]['n_params']:,} | Train={np.mean(tm)/60:.1f}m | "
          f"Inf={np.mean(inf):.2f}ms | "
          f"{f'{np.mean(fl)/1e9:.2f}GFLOPs' if fl else 'N/A'} "
          f"| seeds={[r['seed'] for r in rs]}")

print(f"=== v7 whole-brain Vision Mamba — {TAG} augmentation, 200 subjects ===")
print(f"    native 256^3, {(BRAIN_SIZE//8)**3:,} tokens, brain-masked (mask.mgz)")
print(f"    seeds {INCLUDE_SEEDS}\n")
for k, n in [('mri', 'MRI-only  '), ('pet', 'PET-only  '), ('mm', 'Multimodal')]:
    summarize(results[k], n)

print("\nNOTE: whole-brain streams from disk; ROI models train from an in-memory")

with open(RESULTS, 'w') as f:
    json.dump({k: [{kk: (float(vv) if isinstance(vv, (float, np.floating)) else vv)
                    for kk, vv in r.items()} for r in v] for k, v in results.items()},
              f, indent=2)
print(f"\nsaved {RESULTS} (all seeds retained)")

In [ ]:
NAMES = ['L-Hippo', 'R-Hippo', 'L-Cereb-WM',
         'R-Cereb-WM', 'L-Cerebral-WM', 'R-Cerebral-WM']
UNIFORM = 1 / 6


@torch.no_grad()
def region_weights(model_cls, loaders, mm, prefix, seeds=(1, 7, 123)):
    _, _, te = loaders
    acc = {s: ([], []) if mm else [] for s in seeds}

    for s in seeds:
        path = f"{CKPT_DIR}/{prefix}_seed{s}.pt"
        if not os.path.exists(path):
            print(f"  (no checkpoint for seed {s})"); continue
        m = model_cls(n_rois=6, d_model=32, n_layers=2,
                      n_classes=2, dropout=0.4).to(device)
        m.load_state_dict(torch.load(path, weights_only=True))
        m.eval()
        for batch in te:
            if mm:
                a, b, _, _ = batch
                _, mw, pw = m(a.to(device), b.to(device), return_weights=True)
                acc[s][0].append(mw.cpu().numpy())
                acc[s][1].append(pw.cpu().numpy())
            else:
                a, _, _ = batch
                _, w = m(a.to(device), return_weights=True)
                acc[s].append(w.cpu().numpy())
        del m
    torch.cuda.empty_cache()

    def report(arrays, label):
        rows = []
        for i, nm in enumerate(NAMES):
            per_seed = [a[:, i].mean() for a in arrays]
            rows.append((nm, float(np.mean(per_seed)), per_seed))
        rows.sort(key=lambda r: -r[1])

        print(f"\n  {label}")
        print(f"    {'region':16s} {'attention':>10s} {'vs uniform':>12s}   per-seed")
        for nm, mean, per_seed in rows:
            dev = (mean / UNIFORM - 1) * 100
            seeds_str = ', '.join(f'{v*100:.1f}' for v in per_seed)
            print(f"    {nm:16s} {mean*100:9.1f}% {dev:+11.1f}%   {seeds_str}")
        spread = max(r[1] for r in rows) - min(r[1] for r in rows)
        print(f"    spread {spread*100:.1f} percentage points "
              f"({'concentrated' if spread > 0.10 else 'near-uniform'})")

    print(f"\n{prefix}  (uniform baseline = {UNIFORM*100:.1f}% per region)")
    if mm:
        for j, tag in enumerate(('MRI branch', 'PET branch')):
            arrays = [np.concatenate(acc[s][j]) for s in seeds if acc[s][j]]
            if arrays: report(arrays, tag)
    else:
        arrays = [np.concatenate(acc[s]) for s in seeds if acc[s]]
        if arrays: report(arrays, 'single modality')


region_weights(MultimodalVisionMambaModel, mm_loaders, True, f"{PREFIX}_mm")
region_weights(VisionMambaModel, mri_loaders, False, f"{PREFIX}_mri")
region_weights(VisionMambaModel, pet_loaders, False, f"{PREFIX}_pet")